In [6]:
from __future__ import annotations

import atexit
import warnings
import zipfile
from dataclasses import dataclass
from math import sqrt
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from pmdarima import auto_arima
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, acorr_ljungbox, het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.api import VAR
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss, pacf
from statsmodels.tsa.vector_ar.vecm import coint_johansen

warnings.filterwarnings("ignore")

REQUIRED_FILES = [
    "secondary_data_annual.csv",
    "secondary_data_cleaned.csv",
    "sentiment_data_processed/macro_quarterly_prepared.csv",
    "sentiment_data_processed/sentiment_quarterly_cleaned_by_topic.csv",
    "sentiment_data_processed/model_ready_quarterly_with_scaled.csv",
]

REPORT_PATH = Path.cwd() / "ordered_modeling_report.txt"
_REPORT_FH = open(REPORT_PATH, "w", encoding="utf-8")


def _close_report() -> None:
    if not _REPORT_FH.closed:
        _REPORT_FH.close()


atexit.register(_close_report)


@dataclass
class ProjectSource:
    mode: str
    location: Path
    zip_prefix: str | None = None

    @property
    def label(self) -> str:
        if self.mode == "folder":
            return str(self.location)
        prefix = self.zip_prefix or ""
        return f"{self.location}::{prefix}"

    def exists(self, rel_path: str) -> bool:
        if self.mode == "folder":
            return (self.location / rel_path).exists()

        target = f"{self.zip_prefix or ''}{rel_path}".replace("\\", "/")
        with zipfile.ZipFile(self.location) as zf:
            return target in set(zf.namelist())

    def read_csv(self, rel_path: str, **kwargs) -> pd.DataFrame:
        if self.mode == "folder":
            return pd.read_csv(self.location / rel_path, **kwargs)

        target = f"{self.zip_prefix or ''}{rel_path}".replace("\\", "/")
        with zipfile.ZipFile(self.location) as zf:
            with zf.open(target) as f:
                return pd.read_csv(f, **kwargs)


def emit(*parts, sep: str = " ", end: str = "\n") -> None:
    text = sep.join(str(p) for p in parts)
    print(text, end=end)
    _REPORT_FH.write(text + end)
    _REPORT_FH.flush()


def section(number: int, title: str) -> None:
    emit("\n" + "=" * 110)
    emit(f"{number}. {title}")
    emit("=" * 110)


def subsection(title: str) -> None:
    emit("\n" + title)
    emit("-" * len(title))


def emit_df(df: pd.DataFrame, index: bool = False) -> None:
    if df.empty:
        emit("(không có dữ liệu)")
    else:
        emit(df.to_string(index=index))


@dataclass
class LoadedData:
    source_label: str
    annual: pd.DataFrame
    quarterly: pd.DataFrame
    macro_q: pd.DataFrame
    sent_q: pd.DataFrame
    model_ready: pd.DataFrame
    sent_all: pd.DataFrame
    actual_overlap: pd.DataFrame


@dataclass
class AuditInfo:
    annual_complete: pd.DataFrame
    pre_2019: pd.DataFrame
    coverage_df: pd.DataFrame
    leakage_df: pd.DataFrame


# --------------------------------------------------------------------------------------
# Utility helpers
# --------------------------------------------------------------------------------------

def resolve_zip_prefix(zip_path: Path) -> str | None:
    with zipfile.ZipFile(zip_path) as zf:
        names = set(zf.namelist())
        matches = [name for name in names if name.endswith(REQUIRED_FILES[0])]
        for match in matches:
            prefix = match[: -len(REQUIRED_FILES[0])]
            ok = all(f"{prefix}{rel}".replace("\\", "/") in names for rel in REQUIRED_FILES)
            if ok:
                return prefix
    return None


def resolve_project_source() -> ProjectSource:
    cwd = Path.cwd().resolve()
    try:
        script_dir = Path(__file__).resolve().parent
    except NameError:
        script_dir = cwd
    home = Path.home().resolve()
    project_name = "Data_Science_Midterm_Project-main"
    zip_name = f"{project_name}.zip"

    folder_candidates = [
        cwd,
        cwd / project_name,
        cwd.parent,
        cwd.parent / project_name,
        script_dir,
        script_dir / project_name,
        script_dir.parent,
        script_dir.parent / project_name,
        home / "Downloads",
        home / "Downloads" / project_name,
        home / "Documents",
        home / "Documents" / project_name,
        home / "Desktop",
        home / "Desktop" / project_name,
    ]

    seen = set()
    unique_folders = []
    for p in folder_candidates:
        try:
            p = p.resolve()
        except Exception:
            continue
        if str(p) not in seen:
            seen.add(str(p))
            unique_folders.append(p)

    for folder in unique_folders:
        if folder.exists() and all((folder / rel).exists() for rel in REQUIRED_FILES):
            return ProjectSource(mode="folder", location=folder)

    zip_candidates = [
        cwd / zip_name,
        cwd.parent / zip_name,
        script_dir / zip_name,
        script_dir.parent / zip_name,
        home / "Downloads" / zip_name,
        home / "Documents" / zip_name,
        home / "Desktop" / zip_name,
    ]
    for zp in zip_candidates:
        if zp.exists():
            prefix = resolve_zip_prefix(zp)
            if prefix is not None:
                return ProjectSource(mode="zip", location=zp, zip_prefix=prefix)

    raise FileNotFoundError(
        "Không tìm thấy project folder hoặc file zip chứa dữ liệu.\n"
        f"Thư mục hiện tại: {cwd}\n"
        "Hãy đặt một trong hai thứ sau ở gần notebook/script:\n"
        f"- folder: {project_name}\n"
        f"- zip: {zip_name}"
    )


def safe_first(series: pd.Series):
    series = series.dropna()
    if series.empty:
        return None
    return float(series.iloc[0]) if np.issubdtype(series.dtype, np.number) else series.iloc[0]


def metrics(actual: pd.Series | np.ndarray, pred: np.ndarray) -> tuple[float, float, float]:
    actual_arr = np.asarray(actual, dtype=float)
    pred_arr = np.asarray(pred, dtype=float)
    rmse = sqrt(mean_squared_error(actual_arr, pred_arr))
    mae = mean_absolute_error(actual_arr, pred_arr)
    denom = np.where(np.abs(actual_arr) < 1e-8, np.nan, np.abs(actual_arr))
    mape = np.nanmean(np.abs((actual_arr - pred_arr) / denom)) * 100
    return float(rmse), float(mae), float(mape)


def dm_test(e1: np.ndarray, e2: np.ndarray) -> tuple[float, float]:
    d = (e1**2) - (e2**2)
    T = len(d)
    mean_d = np.mean(d)
    var_d = np.var(d, ddof=1)
    dm_stat = mean_d / np.sqrt(var_d / T)
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return float(dm_stat), float(p_value)


def aicc(res) -> float:
    n = float(res.nobs)
    k = float(res.df_model + 1)
    denom = n - k - 1
    if denom <= 0:
        return float("nan")
    return float(res.aic + (2 * k * (k + 1)) / denom)


def summarize_significant(res, alpha: float = 0.05, ignore: tuple[str, ...] = ("const", "intercept")) -> pd.DataFrame:
    rows = []
    params = getattr(res, "params", None)
    pvalues = getattr(res, "pvalues", None)
    if params is None or pvalues is None:
        return pd.DataFrame(columns=["variable", "coef", "p_value"])
    if hasattr(params, "index"):
        names = list(params.index)
    else:
        names = [str(i) for i in range(len(params))]
    for name in names:
        if name.lower() in ignore:
            continue
        p = float(np.asarray(pvalues[name]).squeeze())
        if np.isfinite(p) and p < alpha:
            coef = float(np.asarray(params[name]).squeeze())
            rows.append({"variable": name, "coef": round(coef, 4), "p_value": round(p, 6)})
    return pd.DataFrame(rows)


def coefficient_table(res, exclude: tuple[str, ...] = ()) -> pd.DataFrame:
    params = getattr(res, "params", None)
    bse = getattr(res, "bse", None)
    pvalues = getattr(res, "pvalues", None)
    tvalues = getattr(res, "tvalues", None)
    if params is None or bse is None or pvalues is None:
        return pd.DataFrame(columns=["variable", "coef", "std_err", "t_or_z", "p_value", "ci_low", "ci_high"])

    if hasattr(params, "index"):
        names = list(params.index)
    else:
        names = [str(i) for i in range(len(params))]

    conf_int = None
    if hasattr(res, "conf_int"):
        try:
            conf_int = res.conf_int()
        except Exception:
            conf_int = None

    rows = []
    for name in names:
        if str(name) in exclude:
            continue
        coef = float(np.asarray(params[name]).squeeze())
        se = float(np.asarray(bse[name]).squeeze())
        stat = float(np.asarray(tvalues[name]).squeeze()) if tvalues is not None else np.nan
        p = float(np.asarray(pvalues[name]).squeeze())
        ci_low = np.nan
        ci_high = np.nan
        if conf_int is not None:
            try:
                ci_low = float(np.asarray(conf_int.loc[name]).squeeze()[0])
                ci_high = float(np.asarray(conf_int.loc[name]).squeeze()[1])
            except Exception:
                try:
                    ci_low = float(conf_int.loc[name, 0])
                    ci_high = float(conf_int.loc[name, 1])
                except Exception:
                    pass
        rows.append(
            {
                "variable": str(name),
                "coef": round(coef, 4),
                "std_err": round(se, 4),
                "t_or_z": round(stat, 4),
                "p_value": round(p, 6),
                "ci_low": round(ci_low, 4) if np.isfinite(ci_low) else np.nan,
                "ci_high": round(ci_high, 4) if np.isfinite(ci_high) else np.nan,
            }
        )
    return pd.DataFrame(rows)


def format_linear_equation(params, y_name: str = "y_hat", exclude: tuple[str, ...] = ()) -> str:
    if hasattr(params, "items"):
        items = list(params.items())
    else:
        items = list(enumerate(params))

    const_names = {"const", "intercept"}
    intercept = 0.0
    pieces = []
    for name, value in items:
        name_str = str(name)
        value_f = float(np.asarray(value).squeeze())
        if name_str.lower() in const_names:
            intercept = value_f
            continue
        if name_str in exclude:
            continue
        sign = "+" if value_f >= 0 else "-"
        pieces.append(f" {sign} {abs(value_f):.4f}*{name_str}")
    return f"{y_name} = {intercept:.4f}" + "".join(pieces)


def format_sarimax_equation(res, y_name: str = "y_t") -> str:
    params = getattr(res, "params", None)
    if params is None:
        return f"{y_name} = <unavailable>"

    if hasattr(params, "items"):
        items = list(params.items())
    else:
        items = list(enumerate(params))

    intercept = 0.0
    exog_terms = []
    ar_terms = []
    ma_terms = []
    for name, value in items:
        name_str = str(name)
        value_f = float(np.asarray(value).squeeze())
        if name_str.lower() in {"const", "intercept"}:
            intercept = value_f
        elif name_str.startswith("ar.L"):
            lag = name_str.split("L", 1)[1]
            sign = "+" if value_f >= 0 else "-"
            ar_terms.append(f" {sign} {abs(value_f):.4f}*{y_name[:-2]}_(t-{lag})")
        elif name_str.startswith("ma.L"):
            lag = name_str.split("L", 1)[1]
            sign = "+" if value_f >= 0 else "-"
            ma_terms.append(f" {sign} {abs(value_f):.4f}*eps_(t-{lag})")
        elif name_str == "sigma2":
            continue
        else:
            sign = "+" if value_f >= 0 else "-"
            exog_terms.append(f" {sign} {abs(value_f):.4f}*{name_str}")

    return f"{y_name} = {intercept:.4f}" + "".join(exog_terms + ar_terms + ma_terms) + " + eps_t"


# --------------------------------------------------------------------------------------
# Data loading and audit
# --------------------------------------------------------------------------------------

def load_data() -> LoadedData:
    source = resolve_project_source()
    annual = source.read_csv("secondary_data_annual.csv")
    quarterly = source.read_csv("secondary_data_cleaned.csv", parse_dates=["Year"]).rename(
        columns={"Year": "quarter_end"}
    )
    macro_q = source.read_csv(
        "sentiment_data_processed/macro_quarterly_prepared.csv",
        parse_dates=["quarter_end"],
    )
    sent_q = source.read_csv(
        "sentiment_data_processed/sentiment_quarterly_cleaned_by_topic.csv",
        parse_dates=["quarter_end"],
    )
    model_ready = source.read_csv(
        "sentiment_data_processed/model_ready_quarterly_with_scaled.csv",
        parse_dates=["quarter_end"],
    )

    sent_all = sent_q[sent_q["topic"] == "all"].copy()
    actual_overlap = (
        macro_q.merge(
            sent_all[
                [
                    "quarter_end",
                    "n_comments",
                    "avg_sentiment",
                    "net_sentiment",
                    "weighted_sentiment_like",
                    "positive_ratio",
                    "negative_ratio",
                ]
            ],
            on="quarter_end",
            how="inner",
        )
        .query("quarter_end >= '2019-09-30' and quarter_end <= '2025-03-31'")
        .copy()
    )

    return LoadedData(
        source_label=source.label,
        annual=annual,
        quarterly=quarterly,
        macro_q=macro_q,
        sent_q=sent_q,
        model_ready=model_ready,
        sent_all=sent_all,
        actual_overlap=actual_overlap,
    )


def build_audit(data: LoadedData) -> AuditInfo:
    annual_complete = data.annual.query("Year <= 2023").dropna().copy()
    pre_2019 = data.model_ready[data.model_ready["quarter_end"] < pd.Timestamp("2019-09-30")].copy()

    coverage_df = pd.DataFrame(
        [
            {
                "dataset": "Annual macro (usable for Model A)",
                "rows": int(annual_complete.shape[0]),
                "range": "1995-2023",
            },
            {
                "dataset": "Quarterly macro",
                "rows": int(data.macro_q.shape[0]),
                "range": f"{data.macro_q['quarter_end'].min().date()} -> {data.macro_q['quarter_end'].max().date()}",
            },
            {
                "dataset": "Quarterly sentiment (topic='all')",
                "rows": int(data.sent_all.shape[0]),
                "range": f"{data.sent_all['quarter_end'].min().date()} -> {data.sent_all['quarter_end'].max().date()}",
            },
            {
                "dataset": "Actual overlap macro + sentiment",
                "rows": int(data.actual_overlap.shape[0]),
                "range": f"{data.actual_overlap['quarter_end'].min().date()} -> {data.actual_overlap['quarter_end'].max().date()}",
            },
        ]
    )

    leakage_rows = []
    for col in ["n_comments", "avg_sentiment", "net_sentiment", "positive_ratio", "negative_ratio"]:
        leakage_rows.append(
            {
                "variable": col,
                "nunique_pre_2019Q3": int(pre_2019[col].nunique(dropna=True)),
                "first_value": safe_first(pre_2019[col]),
            }
        )
    leakage_df = pd.DataFrame(leakage_rows)

    return AuditInfo(
        annual_complete=annual_complete,
        pre_2019=pre_2019,
        coverage_df=coverage_df,
        leakage_df=leakage_df,
    )


# --------------------------------------------------------------------------------------
# Section 14 - Model setup
# --------------------------------------------------------------------------------------

def run_section_14(data: LoadedData, audit: AuditInfo) -> None:
    section(14, "LẬP MÔ HÌNH")
    emit(f"Current working dir: {Path.cwd()}")
    emit(f"Detected data source: {data.source_label}")
    emit(f"Full text output will also be saved to: {REPORT_PATH}")

    subsection("14.1. Khung nghiên cứu")
    emit("- Mô hình A (cấu trúc/kinh tế lượng): giải thích GDP_Growth bằng các biến vĩ mô chính thức.")
    emit("- Mô hình B (forecast/nowcast): kiểm tra giá trị bổ sung của sentiment trên overlap thực 2019Q3-2025Q1.")
    emit("- Các mục 15-20 bên dưới sẽ được in đúng thứ tự lập mô hình trong đề cương.")

    subsection("14.2. Công thức mô hình")
    emit("Model A - OLS/HAC (annual):")
    emit("GDP_Growth_t = beta0 + beta1*Inflation_t + beta2*Interest_Rate_t + beta3*FDI_pct_GDP_t")
    emit("               + beta4*Investment_pct_GDP_t + beta5*Export_pct_GDP_t")
    emit("               + beta6*Unemployment_Rate_t + beta7*Gov_Spending_pct_GDP_t + e_t")
    emit("")
    emit("Model B - Dynamic regression on actual overlap:")
    emit("GDP_Growth_t = alpha0 + alpha1*GDP_Growth_{t-1} + alpha2*Inflation_t + alpha3*Interest_Rate_t")
    emit("               + alpha4*Net_Sentiment_t + alpha5*Avg_Sentiment_t + u_t")
    emit("")
    emit("Forecast model used in section 15:")
    emit("y_t = c + sum(phi_i * y_{t-i}) + sum(theta_j * eps_{t-j}) + gamma'X_t + eps_t")
    emit("with X_t = [Inflation_t, Interest_Rate_t] for macro SARIMAX.")
    emit("")
    emit("Where estimated coefficients will appear in the output:")
    emit("- Model A beta estimates: Section 17.1 (Annual OLS/HAC coefficient table + estimated equation)")
    emit("- Model B alpha estimates: Section 20.1 (dynamic-regression coefficient tables + estimated equations)")
    emit("- Final forecasting coefficients: Section 15.4 (final SARIMAX coefficient table + estimated equation)")

    subsection("14.3. Phạm vi dữ liệu dùng cho từng mô hình")
    emit_df(audit.coverage_df)

    subsection("14.4. Audit leakage / coverage mismatch")
    emit_df(audit.leakage_df)
    emit(f"topic null share before 2019Q3: {round(float(audit.pre_2019['topic'].isna().mean()), 3)}")
    emit(
        "Kết luận audit: KHÔNG dùng model_ready_quarterly_with_scaled.csv để mô hình hóa lịch sử dài hạn,"
    )
    emit(
        "vì sentiment trước 2019Q3 bị backfill bởi cùng một giá trị -> gây leakage."
    )


# --------------------------------------------------------------------------------------
# Section 15 - Forecasting
# --------------------------------------------------------------------------------------

def run_section_15(data: LoadedData) -> dict:
    section(15, "TIME SERIES FORECASTING")

    subsection("15.1. Stationarity (ADF/KPSS) để hỗ trợ chọn d")
    stationarity_rows = []
    for name, series in {
        "log_gdp": data.macro_q["log_gdp"].dropna(),
        "log_gdp_diff1": data.macro_q["log_gdp_diff1"].dropna(),
        "gdp_growth": data.macro_q["gdp_growth"].dropna(),
        "gdp_qoq_pct_recalc": data.macro_q["gdp_qoq_pct_recalc"].dropna(),
        "inflation": data.macro_q["inflation"].dropna(),
        "interest_rate": data.macro_q["interest_rate"].dropna(),
    }.items():
        adf_res = adfuller(series, autolag="AIC")
        try:
            kpss_res = kpss(series, regression="c", nlags="auto")
            kpss_stat, kpss_p = kpss_res[0], kpss_res[1]
        except Exception:
            kpss_stat, kpss_p = np.nan, np.nan
        if (adf_res[1] < 0.05) and (pd.isna(kpss_p) or kpss_p >= 0.05):
            suggestion = "stationary"
        elif name.endswith("diff1"):
            suggestion = "usable after differencing"
        else:
            suggestion = "check differencing / cointegration"
        stationarity_rows.append(
            {
                "series": name,
                "adf_stat": round(adf_res[0], 4),
                "adf_p": round(adf_res[1], 4),
                "kpss_stat": round(float(kpss_stat), 4) if pd.notna(kpss_stat) else np.nan,
                "kpss_p": round(float(kpss_p), 4) if pd.notna(kpss_p) else np.nan,
                "note": suggestion,
            }
        )
    stationarity_df = pd.DataFrame(stationarity_rows)
    emit_df(stationarity_df)

    subsection("15.2. PACF snapshot (first 4 lags)")
    pacf_rows = []
    for target in ["gdp_growth", "gdp_qoq_pct_recalc"]:
        vals = pacf(data.macro_q[target].dropna(), nlags=4, method="ywm")
        row = {"series": target}
        for lag in range(1, 5):
            row[f"pacf_lag{lag}"] = round(float(vals[lag]), 4)
        pacf_rows.append(row)
    emit_df(pd.DataFrame(pacf_rows))

    subsection("15.3. ARIMA vs SARIMAX (train/test theo thời gian)")
    emit("Train end: 2022-12-31 | Test window: 2023Q1-2024Q4 | Exogenous variables: inflation, interest_rate")
    macro_eval = data.macro_q.query("quarter_end <= '2024-12-31'").copy()

    forecast_rows = []
    winner_by_target = {}
    for target in ["log_gdp", "gdp_growth", "gdp_qoq_pct_recalc"]:
        df = macro_eval[["quarter_end", target, "inflation", "interest_rate"]].dropna().copy()
        train_t = df.query("quarter_end <= '2022-12-31'")
        test_t = df.query("quarter_end > '2022-12-31'")

        arima = auto_arima(
            train_t[target],
            seasonal=False,
            information_criterion="aic",
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
            max_p=4,
            max_q=4,
            max_d=2,
        )
        sarimax = auto_arima(
            train_t[target],
            X=train_t[["inflation", "interest_rate"]],
            seasonal=False,
            information_criterion="aic",
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
            max_p=4,
            max_q=4,
            max_d=2,
        )

        pred_arima = arima.predict(n_periods=len(test_t))
        pred_sarimax = sarimax.predict(n_periods=len(test_t), X=test_t[["inflation", "interest_rate"]])

        rmse_a, mae_a, mape_a = metrics(test_t[target], pred_arima)
        rmse_s, mae_s, mape_s = metrics(test_t[target], pred_sarimax)
        winner = "SARIMAX" if rmse_s < rmse_a else "ARIMA"
        winner_by_target[target] = winner

        forecast_rows.extend(
            [
                {
                    "target": target,
                    "model": "ARIMA",
                    "order": str(arima.order),
                    "exog": "no",
                    "RMSE": round(rmse_a, 4),
                    "MAE": round(mae_a, 4),
                    "MAPE": round(mape_a, 2),
                    "winner_by_RMSE": winner,
                },
                {
                    "target": target,
                    "model": "SARIMAX",
                    "order": str(sarimax.order),
                    "exog": "inflation + interest_rate",
                    "RMSE": round(rmse_s, 4),
                    "MAE": round(mae_s, 4),
                    "MAPE": round(mape_s, 2),
                    "winner_by_RMSE": winner,
                },
            ]
        )
    forecast_df = pd.DataFrame(forecast_rows)
    emit_df(forecast_df)

    subsection("15.4. Final macro forecasting model")
    final_df = data.macro_q[["gdp_qoq_pct_recalc", "inflation", "interest_rate"]].dropna().copy()
    final_auto = auto_arima(
        final_df["gdp_qoq_pct_recalc"],
        X=final_df[["inflation", "interest_rate"]],
        seasonal=False,
        suppress_warnings=True,
        error_action="ignore",
        stepwise=True,
        max_p=4,
        max_q=4,
        max_d=2,
    )
    final_sarimax = SARIMAX(
        final_df["gdp_qoq_pct_recalc"],
        exog=final_df[["inflation", "interest_rate"]],
        order=final_auto.order,
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)
    emit(f"Chosen final macro model: SARIMAX{final_auto.order} on gdp_qoq_pct_recalc with inflation + interest_rate")
    emit(final_sarimax.summary().as_text())
    emit("Estimated SARIMAX equation:")
    emit(format_sarimax_equation(final_sarimax, y_name="gdp_qoq_pct_recalc_t"))
    emit("Final SARIMAX coefficient table:")
    emit_df(coefficient_table(final_sarimax, exclude=("sigma2",)))
    lb_df = acorr_ljungbox(final_sarimax.resid.dropna(), lags=[4, 8], return_df=True).round(6)
    emit("Ljung-Box residual test (lags 4, 8):")
    emit_df(lb_df, index=True)
    emit("Significant final-model parameters (p < 0.05):")
    emit_df(summarize_significant(final_sarimax, ignore=("intercept",)))

    return {
        "stationarity_df": stationarity_df,
        "forecast_df": forecast_df,
        "winner_by_target": winner_by_target,
        "final_auto_order": final_auto.order,
        "final_sarimax": final_sarimax,
        "final_ljungbox": lb_df,
    }


# --------------------------------------------------------------------------------------
# Section 16 - VAR
# --------------------------------------------------------------------------------------

def run_section_16(data: LoadedData) -> dict:
    section(16, "VAR (VECTOR AUTOREGRESSION)")
    emit("VAR system: Y_t = c + A1*Y_{t-1} + ... + Ap*Y_{t-p} + u_t")
    emit("where Y_t = [gdp_growth_t, inflation_t, interest_rate_t]'.")

    subsection("16.1. Johansen cointegration")
    vecm_df = data.macro_q[["log_gdp", "inflation", "interest_rate"]].dropna().copy()
    joh = coint_johansen(vecm_df, det_order=0, k_ar_diff=1)
    joh_df = pd.DataFrame(
        {
            "trace_stat": np.round(joh.lr1, 4),
            "crit_90": np.round(joh.cvt[:, 0], 4),
            "crit_95": np.round(joh.cvt[:, 1], 4),
            "crit_99": np.round(joh.cvt[:, 2], 4),
        },
        index=["r = 0", "r <= 1", "r <= 2"],
    )
    emit_df(joh_df, index=True)

    subsection("16.2. Lag selection and VAR fit")
    var_df = data.macro_q[["gdp_growth", "inflation", "interest_rate"]].dropna().copy()
    var_model = VAR(var_df)
    lag_selection = var_model.select_order(maxlags=8)
    emit(lag_selection.summary().as_text())
    selected_lag = int(lag_selection.selected_orders.get("aic", lag_selection.aic))
    var_res = var_model.fit(selected_lag)
    emit(f"Selected lag by AIC: {selected_lag}")
    emit(f"VAR stability (statsmodels.is_stable): {var_res.is_stable(verbose=False)}")
    emit(f"Whiteness p-value (nlags=8): {round(float(var_res.test_whiteness(nlags=8).pvalue), 4)}")
    emit(f"Max |root|: {round(float(np.max(np.abs(var_res.roots))), 4)}")

    subsection("16.3. Granger causality")
    granger_df = pd.DataFrame(
        [
            {
                "hypothesis": "inflation -> gdp_growth",
                "p_value": round(float(var_res.test_causality("gdp_growth", ["inflation"], kind="f").pvalue), 4),
            },
            {
                "hypothesis": "interest_rate -> gdp_growth",
                "p_value": round(float(var_res.test_causality("gdp_growth", ["interest_rate"], kind="f").pvalue), 4),
            },
            {
                "hypothesis": "inflation + interest_rate -> gdp_growth",
                "p_value": round(
                    float(var_res.test_causality("gdp_growth", ["inflation", "interest_rate"], kind="f").pvalue),
                    4,
                ),
            },
        ]
    )
    emit_df(granger_df)

    return {
        "joh_df": joh_df,
        "selected_lag": selected_lag,
        "var_res": var_res,
        "granger_df": granger_df,
    }


# --------------------------------------------------------------------------------------
# Section 17 - OLS/GLS
# --------------------------------------------------------------------------------------

def run_section_17(data: LoadedData, audit: AuditInfo) -> dict:
    section(17, "HỒI QUY ĐA BIẾN (OLS/GLS)")
    emit("Primary structural equation (Model A):")
    emit("GDP_Growth_t = beta0 + beta1*Inflation_t + beta2*Interest_Rate_t + beta3*FDI_pct_GDP_t")
    emit("               + beta4*Investment_pct_GDP_t + beta5*Export_pct_GDP_t")
    emit("               + beta6*Unemployment_Rate_t + beta7*Gov_Spending_pct_GDP_t + e_t")

    X_cols_annual = [
        "Inflation",
        "Interest_Rate",
        "FDI_pct_GDP",
        "Investment_pct_GDP",
        "Export_pct_GDP",
        "Unemployment_Rate",
        "Gov_Spending_pct_GDP",
    ]
    ann = audit.annual_complete.copy()
    X_ann = sm.add_constant(ann[X_cols_annual])
    y_ann = ann["GDP_Growth"]
    ols_hac = sm.OLS(y_ann, X_ann).fit(cov_type="HAC", cov_kwds={"maxlags": 1})

    subsection("17.1. Annual OLS/HAC")
    emit(ols_hac.summary().as_text())
    emit("Model A - coefficient table (these are the estimated beta values):")
    emit_df(coefficient_table(ols_hac))
    emit("Model A - estimated equation:")
    emit(format_linear_equation(ols_hac.params, y_name="GDP_Growth_hat_t"))
    bp = het_breuschpagan(ols_hac.resid, ols_hac.model.exog)
    bg = acorr_breusch_godfrey(sm.OLS(y_ann, X_ann).fit(), nlags=1)
    vif_df = pd.DataFrame(
        {
            "variable": X_ann.columns,
            "VIF": [round(float(variance_inflation_factor(X_ann.values, i)), 3) for i in range(X_ann.shape[1])],
        }
    )
    emit(f"Breusch-Pagan p-value: {round(float(bp[1]), 4)}")
    emit(f"Breusch-Godfrey(1) p-value: {round(float(bg[1]), 4)}")
    emit("Significant annual coefficients (p < 0.05):")
    emit_df(summarize_significant(ols_hac))
    emit("Variance Inflation Factor (VIF):")
    emit_df(vif_df)

    subsection("17.2. Quarterly GLSAR robustness check")
    emit("Caution: quarterly series below are interpolated from annual data, so treat this as robustness only.")
    q_gls = data.quarterly.dropna().copy()
    X_cols_q = [
        "Inflation",
        "Interest_Rate",
        "FDI_pct_GDP",
        "Investment_pct_GDP",
        "Export_pct_GDP",
        "Unemployment_Rate",
        "Gov_Spending_pct_GDP",
    ]
    X_q = sm.add_constant(q_gls[X_cols_q])
    y_q = q_gls["GDP_Growth"]
    glsar = sm.GLSAR(y_q, X_q, rho=1)
    glsar_res = glsar.iterative_fit(maxiter=10)
    emit(glsar_res.summary().as_text())
    emit("Quarterly GLSAR coefficient table:")
    emit_df(coefficient_table(glsar_res))
    emit("Quarterly GLSAR estimated equation:")
    emit(format_linear_equation(glsar_res.params, y_name="GDP_Growth_hat_t"))
    emit(f"Estimated rho: {glsar.rho}")
    emit("Significant quarterly GLSAR coefficients (p < 0.05):")
    emit_df(summarize_significant(glsar_res))

    return {
        "ols_hac": ols_hac,
        "glsar_res": glsar_res,
        "bp_p": float(bp[1]),
        "bg_p": float(bg[1]),
        "vif_df": vif_df,
    }


# --------------------------------------------------------------------------------------
# Section 18 - Prophet
# --------------------------------------------------------------------------------------

def run_section_18(data: LoadedData) -> dict:
    section(18, "PROPHET")
    emit("Prophet is treated as an optional baseline for fast trend forecasting with extra regressors.")
    try:
        from prophet import Prophet  # type: ignore

        prophet_available = True
        err = None
    except Exception as e:  # pragma: no cover
        prophet_available = False
        err = e

    if not prophet_available:
        emit(f"Prophet available: NO -> {repr(err)}")
        emit("Section 18 output = feasibility note only. Prophet is skipped in this environment.")
        return {"available": False, "error": repr(err)}

    prophet_df = data.macro_q[["quarter_end", "gdp_qoq_pct_recalc", "inflation", "interest_rate"]].dropna().copy()
    prophet_df = prophet_df.rename(columns={"quarter_end": "ds", "gdp_qoq_pct_recalc": "y"})
    train_p = prophet_df[prophet_df["ds"] <= pd.Timestamp("2022-12-31")].copy()
    test_p = prophet_df[prophet_df["ds"] > pd.Timestamp("2022-12-31")].copy()

    model = Prophet(yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False)
    model.add_regressor("inflation")
    model.add_regressor("interest_rate")
    model.fit(train_p)
    future = test_p[["ds", "inflation", "interest_rate"]].copy()
    forecast = model.predict(future)
    rmse, mae, mape = metrics(test_p["y"], forecast["yhat"].to_numpy())

    emit("Prophet available: YES")
    emit(f"Prophet holdout metrics on gdp_qoq_pct_recalc -> RMSE={rmse:.4f}, MAE={mae:.4f}, MAPE={mape:.2f}")
    show_df = pd.DataFrame(
        {
            "ds": test_p["ds"].dt.date.to_list(),
            "actual": np.round(test_p["y"].to_numpy(), 4),
            "prophet_pred": np.round(forecast["yhat"].to_numpy(), 4),
        }
    )
    emit_df(show_df)
    return {"available": True, "rmse": rmse, "mae": mae, "mape": mape}

# --------------------------------------------------------------------------------------
# Section 20 - Model choice / final recommendation
# --------------------------------------------------------------------------------------

def run_section_20(data: LoadedData, audit: AuditInfo, s15: dict, s17: dict) -> dict:
    section(20, "LÝ DO CHỌN MÔ HÌNH")
    emit("Section 20 consolidates the evidence used to choose Model A and Model B.")

    subsection("20.1. In-sample dynamic regression on actual overlap")
    emit("Model B baseline formula (macro-only):")
    emit("GDP_Growth_t = alpha0 + alpha1*GDP_Growth_{t-1} + alpha2*inflation_t + alpha3*interest_rate_t + u_t")
    emit("Model B augmented formula (macro + sentiment):")
    emit("GDP_Growth_t = alpha0 + alpha1*GDP_Growth_{t-1} + alpha2*inflation_t + alpha3*interest_rate_t")
    emit("               + alpha4*net_sentiment_t + alpha5*avg_sentiment_t + u_t")

    ov = data.actual_overlap.copy()
    ov["gdp_growth_lag1"] = ov["gdp_growth"].shift(1)
    ov = ov.dropna().copy()

    X_macro = sm.add_constant(ov[["gdp_growth_lag1", "inflation", "interest_rate"]])
    X_comb = sm.add_constant(
        ov[["gdp_growth_lag1", "inflation", "interest_rate", "net_sentiment", "avg_sentiment"]]
    )
    y_ov = ov["gdp_growth"]

    macro_dyn = sm.OLS(y_ov, X_macro).fit(cov_type="HAC", cov_kwds={"maxlags": 1})
    comb_dyn = sm.OLS(y_ov, X_comb).fit(cov_type="HAC", cov_kwds={"maxlags": 1})
    macro_dyn_nr = sm.OLS(y_ov, X_macro).fit()
    comb_dyn_nr = sm.OLS(y_ov, X_comb).fit()

    nested_ftest = comb_dyn_nr.compare_f_test(macro_dyn_nr)
    R = np.zeros((2, len(comb_dyn.params)))
    params = list(comb_dyn.params.index)
    for i, name in enumerate(["net_sentiment", "avg_sentiment"]):
        R[i, params.index(name)] = 1
    wald = comb_dyn.wald_test(R)
    wald_p = float(np.asarray(wald.pvalue).squeeze())

    insample_df = pd.DataFrame(
        [
            {
                "model": "Macro-only dynamic regression",
                "Adj_R2": round(float(macro_dyn.rsquared_adj), 4),
                "AICc": round(aicc(macro_dyn_nr), 4),
            },
            {
                "model": "Macro + sentiment dynamic regression",
                "Adj_R2": round(float(comb_dyn.rsquared_adj), 4),
                "AICc": round(aicc(comb_dyn_nr), 4),
            },
        ]
    )
    emit_df(insample_df)
    emit("Model B macro-only - coefficient table (these are the estimated alpha values):")
    emit_df(coefficient_table(macro_dyn))
    emit("Model B macro-only - estimated equation:")
    emit(format_linear_equation(macro_dyn.params, y_name="GDP_Growth_hat_t"))
    emit("Model B macro+sentiment - coefficient table (these are the estimated alpha values):")
    emit_df(coefficient_table(comb_dyn))
    emit("Model B macro+sentiment - estimated equation:")
    emit(format_linear_equation(comb_dyn.params, y_name="GDP_Growth_hat_t"))
    emit(f"Nested F-test p-value (sentiment adds explanatory power): {round(float(nested_ftest[1]), 4)}")
    emit(f"HAC Wald p-value (joint sentiment significance): {round(wald_p, 6)}")

    subsection("20.2. Rolling one-step forecast: macro-only vs macro+sentiment")
    rolling_rows = []
    rolling_detail = {}
    for target in ["gdp_growth", "gdp_qoq_pct_recalc"]:
        df = data.actual_overlap[
            ["quarter_end", target, "inflation", "interest_rate", "net_sentiment", "avg_sentiment"]
        ].dropna().reset_index(drop=True)

        init_train = 12
        auto_macro = auto_arima(
            df.loc[: init_train - 1, target],
            X=df.loc[: init_train - 1, ["inflation", "interest_rate"]],
            seasonal=False,
            suppress_warnings=True,
            error_action="ignore",
            stepwise=True,
            max_p=2,
            max_q=2,
            max_d=1,
            information_criterion="aic",
        )
        auto_comb = auto_arima(
            df.loc[: init_train - 1, target],
            X=df.loc[: init_train - 1, ["inflation", "interest_rate", "net_sentiment", "avg_sentiment"]],
            seasonal=False,
            suppress_warnings=True,
            error_action="ignore",
            stepwise=True,
            max_p=2,
            max_q=2,
            max_d=1,
            information_criterion="aic",
        )

        actual_vals, pred_macro_vals, pred_comb_vals, forecast_dates = [], [], [], []
        for i in range(init_train, len(df)):
            tr = df.iloc[:i].copy()
            te = df.iloc[i : i + 1].copy()

            m1 = SARIMAX(
                tr[target],
                exog=tr[["inflation", "interest_rate"]],
                order=auto_macro.order,
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False)
            m2 = SARIMAX(
                tr[target],
                exog=tr[["inflation", "interest_rate", "net_sentiment", "avg_sentiment"]],
                order=auto_comb.order,
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False)

            p1 = m1.get_forecast(steps=1, exog=te[["inflation", "interest_rate"]]).predicted_mean.iloc[0]
            p2 = m2.get_forecast(
                steps=1,
                exog=te[["inflation", "interest_rate", "net_sentiment", "avg_sentiment"]],
            ).predicted_mean.iloc[0]

            actual_vals.append(te[target].iloc[0])
            pred_macro_vals.append(p1)
            pred_comb_vals.append(p2)
            forecast_dates.append(te["quarter_end"].iloc[0].date())

        actual_arr = np.asarray(actual_vals, dtype=float)
        pred_macro_arr = np.asarray(pred_macro_vals, dtype=float)
        pred_comb_arr = np.asarray(pred_comb_vals, dtype=float)
        rmse_macro, mae_macro, mape_macro = metrics(actual_arr, pred_macro_arr)
        rmse_comb, mae_comb, mape_comb = metrics(actual_arr, pred_comb_arr)
        dm_stat, dm_p = dm_test(actual_arr - pred_macro_arr, actual_arr - pred_comb_arr)

        rolling_rows.append(
            {
                "target": target,
                "macro_order": str(auto_macro.order),
                "macro_sent_order": str(auto_comb.order),
                "RMSE_macro": round(rmse_macro, 4),
                "RMSE_macro_sent": round(rmse_comb, 4),
                "MAE_macro": round(mae_macro, 4),
                "MAE_macro_sent": round(mae_comb, 4),
                "MAPE_macro": round(mape_macro, 2),
                "MAPE_macro_sent": round(mape_comb, 2),
                "DM_p_value": round(dm_p, 4),
                "better_by_RMSE": "macro-only" if rmse_macro < rmse_comb else "macro+sentiment",
            }
        )
        rolling_detail[target] = pd.DataFrame(
            {
                "quarter_end": forecast_dates,
                "actual": np.round(actual_arr, 4),
                "pred_macro": np.round(pred_macro_arr, 4),
                "pred_macro_sent": np.round(pred_comb_arr, 4),
            }
        )
    rolling_df = pd.DataFrame(rolling_rows)
    emit_df(rolling_df)
    for target, cmp_df in rolling_detail.items():
        emit(f"\nForecast detail - {target}")
        emit_df(cmp_df)

    subsection("20.3. Structural break test for the sentiment-augmented overlap model")
    chow_rows = []
    for break_date in ["2021-12-31", "2022-03-31", "2023-03-31"]:
        mask = ov["quarter_end"] <= pd.Timestamp(break_date)
        n1 = int(mask.sum())
        n2 = int((~mask).sum())
        X_full = X_comb
        k = X_full.shape[1]
        if n1 <= k or n2 <= k:
            continue
        pooled = sm.OLS(y_ov, X_full).fit()
        part1 = sm.OLS(y_ov[mask], X_full[mask]).fit()
        part2 = sm.OLS(y_ov[~mask], X_full[~mask]).fit()
        rss_p = float(np.sum(pooled.resid**2))
        rss_1 = float(np.sum(part1.resid**2))
        rss_2 = float(np.sum(part2.resid**2))
        F = ((rss_p - (rss_1 + rss_2)) / k) / ((rss_1 + rss_2) / (n1 + n2 - 2 * k))
        p_value = 1 - stats.f.cdf(F, k, n1 + n2 - 2 * k)
        chow_rows.append(
            {
                "break_date": break_date,
                "F_stat": round(float(F), 4),
                "p_value": round(float(p_value), 8),
                "n1": n1,
                "n2": n2,
            }
        )
    chow_df = pd.DataFrame(chow_rows)
    emit_df(chow_df)

    subsection("20.4. Why Model A and Model B should be separated")
    reason_rows = [
        {
            "criterion": "Coverage",
            "evidence": f"Annual macro usable rows = {audit.annual_complete.shape[0]}, but actual macro+sentiment overlap = {data.actual_overlap.shape[0]} quarters",
            "implication": "Cannot treat long-run macro and short-run sentiment as one homogeneous sample",
        },
        {
            "criterion": "Leakage",
            "evidence": "Sentiment columns in model_ready before 2019Q3 have nunique = 1 -> clear backfill pattern",
            "implication": "Do not use model_ready for historical training",
        },
        {
            "criterion": "Out-of-sample performance",
            "evidence": "Rolling forecasts show macro-only beats or matches macro+sentiment on RMSE for both targets",
            "implication": "Sentiment helps in-sample fit, but not forecasting accuracy",
        },
        {
            "criterion": "Structural stability",
            "evidence": "Chow tests indicate strong break around 2021-2022",
            "implication": "Sentiment relationship is not stable across the overlap period",
        },
        {
            "criterion": "Best forecasting baseline",
            "evidence": f"Section 15 selects SARIMAX{s15['final_auto_order']} for gdp_qoq_pct_recalc",
            "implication": "Use macro SARIMAX as the main short-term forecasting baseline",
        },
        {
            "criterion": "Interpretability",
            "evidence": "Section 17 keeps an annual OLS/HAC model with interpretable macro coefficients",
            "implication": "Use OLS/HAC as the main structural model",
        },
    ]
    reason_df = pd.DataFrame(reason_rows)
    emit_df(reason_df)

    subsection("20.5. Final recommendation")
    emit("1) NÊN TÁCH thành 2 mô hình nghiên cứu dưới cùng một khung đề tài.")
    emit("   - Mô hình A (cấu trúc/kinh tế lượng): annual OLS/HAC để giải thích GDP growth bằng biến vĩ mô chính thức.")
    emit("   - Mô hình B (forecast/nowcast): quarterly SARIMAX làm baseline dự báo; sentiment chỉ dùng như mô hình phụ trợ trên overlap thực.")
    emit("2) Không nên gộp thành một mô hình duy nhất vì:")
    emit("   - coverage mismatch giữa macro dài hạn và sentiment ngắn hạn;")
    emit("   - model_ready có leakage/backfill sentiment trước 2019Q3;")
    emit("   - sentiment cải thiện in-sample nhưng không cải thiện out-of-sample;")
    emit("   - có structural break mạnh quanh 2021-2022.")
    emit("3) Kết luận lựa chọn mô hình:")
    emit("   - Model A main choice: Annual OLS/HAC (explainability / structural interpretation)")
    emit("   - Model B main choice: Quarterly SARIMAX on gdp_qoq_pct_recalc (forecasting)")
    emit("   - Sentiment model: supplementary evidence only, not the single integrated final model")

    return {
        "insample_df": insample_df,
        "rolling_df": rolling_df,
        "chow_df": chow_df,
        "reason_df": reason_df,
    }


# --------------------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------------------

def main() -> None:
    data = load_data()
    audit = build_audit(data)
    run_section_14(data, audit)
    s15 = run_section_15(data)
    _ = run_section_16(data)
    s17 = run_section_17(data, audit)
    _ = run_section_18(data)
    _ = run_section_19(data)
    _ = run_section_20(data, audit, s15, s17)
    emit("\n" + "=" * 110)
    emit(f"DONE. Full report saved to: {REPORT_PATH}")
    emit("=" * 110)


if __name__ == "__main__":
    main()



14. LẬP MÔ HÌNH
Current working dir: c:\Users\Windows\Downloads\Midterm KHDL
Detected data source: C:\Users\Windows\Downloads\Midterm KHDL\Data_Science_Midterm_Project-main
Full text output will also be saved to: c:\Users\Windows\Downloads\Midterm KHDL\ordered_modeling_report.txt

14.1. Khung nghiên cứu
----------------------
- Mô hình A (cấu trúc/kinh tế lượng): giải thích GDP_Growth bằng các biến vĩ mô chính thức.
- Mô hình B (forecast/nowcast): kiểm tra giá trị bổ sung của sentiment trên overlap thực 2019Q3-2025Q1.
- Các mục 15-20 bên dưới sẽ được in đúng thứ tự lập mô hình trong đề cương.

14.2. Công thức mô hình
-----------------------
Model A - OLS/HAC (annual):
GDP_Growth_t = beta0 + beta1*Inflation_t + beta2*Interest_Rate_t + beta3*FDI_pct_GDP_t
               + beta4*Investment_pct_GDP_t + beta5*Export_pct_GDP_t
               + beta6*Unemployment_Rate_t + beta7*Gov_Spending_pct_GDP_t + e_t

Model B - Dynamic regression on actual overlap:
GDP_Growth_t = alpha0 + alpha1*GDP_Gr